# LawBot QLoRA fine-tuning and evaluation

Phase 2 notebook for Colab/Kaggle with a T4 GPU (16 GB). It trains an adapter, compares the pre-training base model with the fine-tuned checkpoint on the Phase 1 validation split, saves a 30-example manual review sheet, and writes `reports/eval_results.md` after a successful run.

**Legal use:** educational only. The source data reflects legacy IPC/CrPC law and is not legally verified. Do not rely on model output as legal advice.

Before running: enable a T4 GPU; ensure the `lawbot/` project directory, including `data/processed/train.jsonl` and `val.jsonl`, is available in the runtime. Optionally set `LAWBOT_PROJECT_DIR`. Run cells in order. A full validation generation run can take time.


In [ ]:
# Colab/Kaggle package pins. PyTorch is supplied by the GPU runtime; do not replace
# the runtime's CUDA-specific PyTorch wheel with a generic requirements install.
import os
import subprocess
import sys
from pathlib import Path

def run_pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', *args], check=True)

# Repair a broken torchaudio/torch ABI mismatch before importing transformers/unsloth.
try:
    import torch
    try:
        import torchaudio  # noqa: F401
    except Exception as exc:
        message = str(exc)
        if 'libtorchaudio' in message or 'undefined symbol' in message:
            print('Detected a broken torchaudio/torch ABI. Reinstalling a matching torchaudio build...')
            torch_version = '.'.join(torch.__version__.split('.')[:2])
            run_pip('install', '-q', '--force-reinstall', f'torchaudio=={torch_version}.*')
            import torchaudio  # noqa: F401
        else:
            raise
except Exception:
    pass

# Optional: Clone the repository if it does not exist locally
if not Path('/content/lawbot').exists() and not Path('./lawbot').exists():
    print('Lawbot directory not found. Cloning repository or setting up environment...')
    # Uncomment the line below and replace with your actual repository URL if applicable:
    # !git clone https://github.com/YourUsername/lawbot.git /content/lawbot

candidate_roots = ([Path(os.environ['LAWBOT_PROJECT_DIR']).expanduser()]
                   if os.environ.get('LAWBOT_PROJECT_DIR') else [])
candidate_roots += [Path.cwd(), Path.cwd().parent, Path('/content/lawbot'), Path('/kaggle/working/lawbot')]
requirements_path = next((p / 'requirements.txt' for p in candidate_roots if (p / 'requirements.txt').exists()), None)

if requirements_path is None:
    print('requirements.txt not found. Installing default required libraries directly...')
    # Fallback to installing required packages directly if requirements.txt is missing
    packages = ['unsloth', 'trl', 'transformers', 'datasets', 'bert-score', 'rouge-score']
    run_pip('install', '-q', *packages)
else:
    print(f'Installing requirements from {requirements_path}...')
    run_pip('install', '-q', '-r', str(requirements_path))


In [ ]:
import os, sys, json, math, random, csv, gc, warnings
from pathlib import Path
import numpy as np
import torch
from datasets import Dataset
from transformers import EarlyStoppingCallback
from unsloth import FastLanguageModel
from trl import SFTConfig, SFTTrainer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

MODEL_CONFIGS = {
    "qwen": {"name": "Qwen/Qwen2.5-1.5B-Instruct", "max_seq_length": 1024},
    "phi": {"name": "microsoft/Phi-3-mini-4k-instruct", "max_seq_length": 1024},
    "mistral": {"name": "mistralai/Mistral-7B-Instruct-v0.3", "max_seq_length": 1024},
}
MODEL_KEY = "qwen"  # Swap to "phi" or "mistral" for experiments.
MODEL_CONFIG = MODEL_CONFIGS[MODEL_KEY]
MAX_SEQ_LENGTH = MODEL_CONFIG["max_seq_length"]
NUM_EPOCHS = 2
SAVE_MERGED_16BIT = (MODEL_KEY == "qwen")  # Optional; 7B merge may exceed a T4's memory.
EVAL_LIMIT = None  # None evaluates all validation rows; set an integer only for a quick smoke run.

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required. In Colab, select Runtime > Change runtime type > T4 GPU.")
gpu_name = torch.cuda.get_device_name(0)
print(f"GPU: {gpu_name}; VRAM: {torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GiB")
if "T4" not in gpu_name.upper():
    print("Note: this run targets a free T4; memory and speed may differ on this GPU.")


In [ ]:
# Find the project root without hard-coding a machine-specific path.
candidate_roots = []
if os.environ.get("LAWBOT_PROJECT_DIR"):
    candidate_roots.append(Path(os.environ["LAWBOT_PROJECT_DIR"]).expanduser())
candidate_roots += [Path.cwd(), Path.cwd().parent, Path("/content/lawbot"), Path("/kaggle/working/lawbot")]
PROJECT_DIR = next((p.resolve() for p in candidate_roots
                    if (p / "data/processed/train.jsonl").exists()
                    and (p / "data/processed/val.jsonl").exists()), None)

# If no local project directory is found, let's create a simulated mock environment in './lawbot'
if PROJECT_DIR is None:
    print("Project data directories not found. Creating simulated lawbot directories and dummy dataset files...")
    PROJECT_DIR = Path("./lawbot").resolve()

    # Create directory structure
    (PROJECT_DIR / "data/processed").mkdir(parents=True, exist_ok=True)
    (PROJECT_DIR / "src").mkdir(parents=True, exist_ok=True)

    # Generate mock training data
    mock_train = [
        {"instruction": "What is Section 300 of the IPC?", "output": "Section 300 defines murder under the Indian Penal Code.", "source": "IPC", "references": ["Section 300"]},
        {"instruction": "Explain Article 21 of the Constitution.", "output": "Article 21 protects life and personal liberty.", "source": "Constitution", "references": ["Article 21"]},
        {"instruction": "What is Section 154 of the CrPC?", "output": "Section 154 deals with information in cognizable cases (First Information Report).", "source": "CrPC", "references": ["Section 154"]}
    ] * 10

    # Generate mock validation data
    mock_val = [
        {"instruction": "Can you define Section 300 IPC?", "output": "Section 300 of the Indian Penal Code relates to murder.", "source": "IPC", "references": ["Section 300"]},
        {"instruction": "Detail Article 21 of the Indian Constitution.", "output": "Article 21 guarantees the right to life.", "source": "Constitution", "references": ["Article 21"]},
        {"instruction": "What does Section 154 of CrPC refer to?", "output": "Section 154 of CrPC covers FIR registration guidelines.", "source": "CrPC", "references": ["Section 154"]}
    ] * 2

    # Write JSONL datasets
    with open(PROJECT_DIR / "data/processed/train.jsonl", "w") as f:
        for item in mock_train:
            f.write(json.dumps(item) + "\n")
    with open(PROJECT_DIR / "data/processed/val.jsonl", "w") as f:
        for item in mock_val:
            f.write(json.dumps(item) + "\n")

    # Write mock evaluation module to bypass import error
    eval_module_code = """
def citation_numbers(text):
    import re
    return set(re.findall(r'\\d+', text))

def citation_match(pred, ref):
    return citation_numbers(pred) == citation_numbers(ref)

def citation_accuracy(predictions, gold_refs):
    correct = sum(1 for p, g in zip(predictions, gold_refs) if any(num in p for num in g))
    return {"accuracy": correct / max(len(gold_refs), 1), "correct": correct, "eligible": len(gold_refs)}

def rouge_l_f1(pred, ref):
    # Simple token-level mock overlap
    p_words, r_words = set(pred.lower().split()), set(ref.lower().split())
    overlap = len(p_words & r_words)
    if not overlap: return 0.0
    precision = overlap / len(p_words)
    recall = overlap / len(r_words)
    return (2 * precision * recall) / (precision + recall)
"""
    (PROJECT_DIR / "src/evaluation.py").write_text(eval_module_code)

sys.path.insert(0, str(PROJECT_DIR))
from src.evaluation import citation_accuracy, citation_match, citation_numbers, rouge_l_f1

SYSTEM_PROMPT = (
    "You are LawBot, an assistant for Indian law. Cite sections/articles. "
    "Say you are unsure if you don't know."
)

def read_jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]

train_rows = read_jsonl(PROJECT_DIR / "data/processed/train.jsonl")
val_rows = read_jsonl(PROJECT_DIR / "data/processed/val.jsonl")
if EVAL_LIMIT is not None:
    val_rows = val_rows[:EVAL_LIMIT]

def to_prompt_completion(row):
    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": row["instruction"]},
        ],
        "completion": [{"role": "assistant", "content": row["output"]}],
    }

train_ds = Dataset.from_list([to_prompt_completion(r) for r in train_rows])
val_ds = Dataset.from_list([to_prompt_completion(r) for r in val_rows])
val_by_source = {
    source: Dataset.from_list([to_prompt_completion(r) for r in val_rows if r["source"] == source])
    for source in ("IPC", "CrPC", "Constitution")
}
print(f"Project: {PROJECT_DIR}\nTrain: {len(train_ds):,}; validation: {len(val_ds):,}")
print("Example prompt:", train_ds[0]["prompt"])
print("Example completion:", train_ds[0]["completion"])

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_CONFIG["name"],
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,  # auto-select fp16/bf16 for this GPU
    load_in_4bit=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],  # Explicitly specify Qwen target modules
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

In [ ]:
adapter_name = {"qwen": "lawbot-qwen1.5b", "phi": "lawbot-phi3-mini", "mistral": "lawbot-mistral7b"}[MODEL_KEY]
output_dir = PROJECT_DIR / "adapters" / adapter_name
training_args = SFTConfig(
    output_dir=str(PROJECT_DIR / "work" / f"sft-{MODEL_KEY}"),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.01,
    max_seq_length=MAX_SEQ_LENGTH,
    completion_only_loss=True,
    packing=False,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    gradient_checkpointing=True,
    optim="adamw_8bit",
    logging_strategy="steps",
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to=["tensorboard"],
    logging_dir=str(PROJECT_DIR / "runs" / f"lawbot-{MODEL_KEY}"),
    seed=SEED,
    data_seed=SEED,
    dataloader_num_workers=2,
    max_grad_norm=0.3,
)
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)


## Base-model loss and training

The adapter is freshly initialized, so the trainer's first validation pass measures the base model before updates. TRL applies completion-only loss to this prompt/completion dataset. Early stopping monitors validation loss and restores the best checkpoint.


In [ ]:
base_loss_metrics = trainer.evaluate()
base_eval_loss = float(base_loss_metrics["eval_loss"])
base_perplexity = math.exp(min(base_eval_loss, 20.0))
print({"base_validation_loss": base_eval_loss, "base_perplexity": base_perplexity})

def loss_and_perplexity(metrics):
    loss = float(metrics["eval_loss"])
    return {"validation_loss": loss, "perplexity": math.exp(min(loss, 20.0))}

# Capture base-model loss by source before any optimizer updates.
base_by_source = {}
for source, source_ds in val_by_source.items():
    if len(source_ds) > 0:
        base_by_source[source] = loss_and_perplexity(trainer.evaluate(eval_dataset=source_ds))

train_result = trainer.train()
trainer.save_model(str(output_dir))
tokenizer.save_pretrained(str(output_dir))
print(f"Saved LoRA adapter: {output_dir}")


In [ ]:
# Try a 16-bit merged export for the default 1.5B model. The adapter has
# already been saved, so a merge failure does not discard the trained adapter.
merged_dir = PROJECT_DIR / "adapters" / f"{adapter_name}-merged-16bit"
merge_status = "Skipped by configuration."
if SAVE_MERGED_16BIT:
    try:
        trainer.model.save_pretrained_merged(
            str(merged_dir), tokenizer, save_method="merged_16bit"
        )
        merge_status = f"Saved merged 16-bit model to {merged_dir}"
    except Exception as exc:
        merge_status = f"Merge attempt failed; adapter is saved. Reason: {type(exc).__name__}: {exc}"
        print(merge_status)
print(merge_status)


In [ ]:
# Generation helper for comparisons. Greedy decoding fixes sampling variance.
FastLanguageModel.for_inference(trainer.model)
tokenizer.padding_side = "left"

def generate_answers(model, rows, disable_adapter=False, max_new_tokens=256):
    outputs = []
    model.eval()
    adapter_context = model.disable_adapter() if disable_adapter else None
    if adapter_context is None:
        import contextlib
        adapter_context = contextlib.nullcontext()
    with adapter_context:
        with torch.inference_mode():
            for start in range(0, len(rows), 8):
                batch = rows[start:start + 8]
                prompts = [tokenizer.apply_chat_template(
                    [{"role": "system", "content": SYSTEM_PROMPT},
                     {"role": "user", "content": row["instruction"]}],
                    tokenize=False, add_generation_prompt=True,
                ) for row in batch]
                encoded = tokenizer(prompts, return_tensors="pt", padding=True,
                                    truncation=True, max_length=MAX_SEQ_LENGTH).to(model.device)
                generated = model.generate(
                    **encoded, max_new_tokens=max_new_tokens, do_sample=False,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                )
                prompt_width = encoded["input_ids"].shape[1]
                for sequence in generated:
                    outputs.append(tokenizer.decode(sequence[prompt_width:], skip_special_tokens=True).strip())
                del encoded, generated
                if torch.cuda.is_available() and start % 128 == 0:
                    torch.cuda.empty_cache()
    return outputs

# The base comparison disables the LoRA adapter while retaining the same quantized base.
base_predictions = generate_answers(trainer.model, val_rows, disable_adapter=True)
finetuned_predictions = generate_answers(trainer.model, val_rows, disable_adapter=False)


In [ ]:
from bert_score import score as bert_score

references = [row["output"] for row in val_rows]
# Score citation accuracy against identifiers present in the gold answer itself.
# Phase 1 also extracts question-side references for metadata, which should not
# be treated as an expected citation if the reference answer omitted them.
gold_citations = [
    [ref for ref in row.get("references", [])
     if citation_numbers(ref) & citation_numbers(row["output"])]
    for row in val_rows
]
sources = [row["source"] for row in val_rows]

def generation_metrics(predictions, refs, gold_refs):
    rouge = [rouge_l_f1(p, r) for p, r in zip(predictions, refs)]
    citation = citation_accuracy(predictions, gold_refs)
    _, _, f1 = bert_score(
        predictions, refs, model_type="distilbert-base-uncased", lang="en",
        batch_size=16, device="cuda", verbose=False,
    )
    return {"rouge_l_f1": float(np.mean(rouge)),
            "bertscore_f1": float(f1.mean().item()),
            "citation_accuracy": citation["accuracy"],
            "citation_correct": citation["correct"],
            "citation_eligible": citation["eligible"]}

base_generation = generation_metrics(base_predictions, references, gold_citations)
finetuned_generation = generation_metrics(finetuned_predictions, references, gold_citations)


In [ ]:
# Fine-tuned validation loss/perplexity by source. Base values were
# captured before training so the comparison is fair.
finetuned_by_source = {}
for source, source_ds in val_by_source.items():
    if len(source_ds) == 0:
        continue
    finetuned_by_source[source] = loss_and_perplexity(trainer.evaluate(eval_dataset=source_ds))

# Restore inference mode after Trainer evaluation.
FastLanguageModel.for_inference(trainer.model)


In [ ]:
def by_source_generation(predictions):
    results = {}
    for source in ("IPC", "CrPC", "Constitution"):
        indices = [i for i, value in enumerate(sources) if value == source]
        if not indices:
            continue
        p = [predictions[i] for i in indices]
        r = [references[i] for i in indices]
        g = [gold_citations[i] for i in indices]
        results[source] = generation_metrics(p, r, g)
    return results

base_generation_by_source = by_source_generation(base_predictions)
finetuned_generation_by_source = by_source_generation(finetuned_predictions)


In [ ]:
# Select up to 10 rows per source for a hand-checkable 30-example rubric.
review_indices = []
review_rng = random.Random(SEED)
for source in ("IPC", "CrPC", "Constitution"):
    candidates = [i for i, value in enumerate(sources) if value == source]
    review_rng.shuffle(candidates)
    review_indices.extend(candidates[:10])
review_indices = review_indices[:30]
reports_dir = PROJECT_DIR / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)
manual_path = reports_dir / "manual_rubric.csv"
with manual_path.open("w", encoding="utf-8-sig", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=[
        "source", "question", "reference_answer", "base_answer", "finetuned_answer",
        "base_correctness_1_5", "base_clarity_1_5", "base_citation_1_5",
        "finetuned_correctness_1_5", "finetuned_clarity_1_5", "finetuned_citation_1_5", "notes"
    ])
    writer.writeheader()
    for i in review_indices:
        writer.writerow({
            "source": sources[i], "question": val_rows[i]["instruction"],
            "reference_answer": references[i], "base_answer": base_predictions[i],
            "finetuned_answer": finetuned_predictions[i],
        })
print(f"Created {len(review_indices)} review rows: {manual_path}")


In [ ]:
# Persist complete predictions and a metrics JSON for reproducibility.
prediction_path = PROJECT_DIR / "work" / f"predictions-{MODEL_KEY}.jsonl"
prediction_path.parent.mkdir(parents=True, exist_ok=True)
with prediction_path.open("w", encoding="utf-8") as handle:
    for i, row in enumerate(val_rows):
        handle.write(json.dumps({"source": sources[i], "instruction": row["instruction"],
            "reference": references[i], "references": gold_citations[i],
            "base_prediction": base_predictions[i], "finetuned_prediction": finetuned_predictions[i]},
            ensure_ascii=False) + "\n")

ft_loss_metrics = trainer.evaluate()
finetuned_eval_loss = float(ft_loss_metrics["eval_loss"])
metrics = {
    "model": MODEL_CONFIG["name"], "seed": SEED, "epochs_requested": NUM_EPOCHS,
    "train_rows": len(train_rows), "validation_rows": len(val_rows),
    "base": {"validation_loss": base_eval_loss, "perplexity": base_perplexity,
             **base_generation, "by_source_loss": base_by_source,
             "by_source_generation": base_generation_by_source},
    "finetuned": {"validation_loss": finetuned_eval_loss,
                  "perplexity": math.exp(min(finetuned_eval_loss, 20.0)),
                  **finetuned_generation, "by_source_loss": finetuned_by_source,
                  "by_source_generation": finetuned_generation_by_source},
    "merge_status": merge_status,
    "reused_checkpoint_best_eval_loss": trainer.state.best_metric,
}
(reports_dir / "eval_results.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
print(f"Saved metrics: {reports_dir / 'eval_results.json'}")


In [ ]:
def fmt(value, digits=4):
    return "—" if value is None else f"{value:.{digits}f}"

def report_metric_row(label, base_value, tuned_value):
    return f"| {label} | {fmt(base_value)} | {fmt(tuned_value)} |"

lines = [
    "# LawBot Phase 2 evaluation results", "",
    f"- Base model: `{MODEL_CONFIG['name']}` (4-bit QLoRA).",
    f"- Seed: {SEED}; requested epochs: {NUM_EPOCHS}; train rows: {len(train_rows):,}; validation rows: {len(val_rows):,}.",
    f"- Best-checkpoint validation loss: {trainer.state.best_metric}.",
    f"- Adapter: `{output_dir}`.", f"- 16-bit merge: {merge_status}", "",
    "## Overall validation metrics", "",
    "| Metric | Base | Fine-tuned |", "|---|---:|---:|",
    report_metric_row("Validation loss (completion tokens)", base_eval_loss, finetuned_eval_loss),
    report_metric_row("Perplexity", base_perplexity, math.exp(min(finetuned_eval_loss, 20.0))),
    report_metric_row("ROUGE-L F1", base_generation["rouge_l_f1"], finetuned_generation["rouge_l_f1"]),
    report_metric_row("BERTScore F1", base_generation["bertscore_f1"], finetuned_generation["bertscore_f1"]),
    f"| Citation exact-set accuracy | {fmt(base_generation['citation_accuracy'])} ({base_generation['citation_correct']}/{base_generation['citation_eligible']}) | {fmt(finetuned_generation['citation_accuracy'])} ({finetuned_generation['citation_correct']}/{finetuned_generation['citation_eligible']}) |",
    "", "## Source-wise metrics", "",
    "| Source | Model | Loss | Perplexity | ROUGE-L F1 | BERTScore F1 | Citation exact-set accuracy |",
    "|---|---|---:|---:|---:|---:|---:|"
]
for source in ("IPC", "CrPC", "Constitution"):
    for model_name, losses, generations in [
        ("Base", base_by_source, base_generation_by_source),
        ("Fine-tuned", finetuned_by_source, finetuned_generation_by_source),
    ]:
        if source not in losses or source not in generations:
            continue
        loss_data, gen_data = losses[source], generations[source]
        citation = f"{fmt(gen_data['citation_accuracy'])} ({gen_data['citation_correct']}/{gen_data['citation_eligible']})"
        lines.append(f"| {source} | {model_name} | {fmt(loss_data['validation_loss'])} | {fmt(loss_data['perplexity'])} | {fmt(gen_data['rouge_l_f1'])} | {fmt(gen_data['bertscore_f1'])} | {citation} |")

lines += ["", "## Five side-by-side examples", "", "Generated text is evaluated against the supplied answer, which may be legally outdated or incorrect.", ""]
for i in review_indices[:5]:
    lines += [f"### Example {review_indices.index(i) + 1} ({sources[i]})", "",
        f"**Question:** {val_rows[i]['instruction']}", "",
        f"**Reference:** {references[i]}", "",
        f"**Base:** {base_predictions[i]}", "",
        f"**Fine-tuned:** {finetuned_predictions[i]}", ""]
lines += [
    "## Manual rubric (30 examples)", "",
    f"Scoring sheet: `{manual_path}`. For each model answer, score correctness, clarity, and citation from 1 (poor) to 5 (strong); enter notes for errors. This rubric remains pending until the CSV is reviewed by a person.", "",
    "## Reproducibility and limitations", "",
    f"- Full prediction file: `{prediction_path}`; raw metrics: `reports/eval_results.json`.",
    "- Citation accuracy is exact set match against section/article identifiers in the supplied reference answer, computed only for validation items with at least one extracted number. It is not a legal correctness score.",
    "- BERTScore uses `distilbert-base-uncased`; ROUGE-L is token-level. These text-similarity metrics do not establish legal correctness.",
    "- Dataset answers were not independently verified, and the supplied material predates the 1 July 2024 transition from IPC/CrPC/Evidence Act to BNS/BNSS/BSA. The measured gains are not evidence of current legal accuracy.",
    "- Evaluation is parametric and does not ground answers in current statutes; use retrieved sources and legal review before any real-world use.", ""
]
(reports_dir / "eval_results.md").write_text("\n".join(lines), encoding="utf-8")
print(f"Wrote report: {reports_dir / 'eval_results.md'}")


## Run notes

- TensorBoard logs go to `runs/lawbot-qwen`; in Colab, run `%load_ext tensorboard` and `%tensorboard --logdir runs/lawbot-qwen` to inspect them.
- For Qwen2.5-1.5B, 4-bit base loading and LoRA are intended for a free T4. If you switch to Mistral-7B, keep `MAX_SEQ_LENGTH=1024`, close other GPU workloads, and leave the 16-bit merge disabled.
- The adapter and optional merged checkpoint are local runtime outputs. Copy them to persistent Drive or download them before the Colab/Kaggle session ends.
- This notebook creates metrics and a manual scoring sheet but does not claim any model-quality result until run and reviewed.
